# Evidence-Grounded AI Framework for Cross-Corpus Innovation Assessment
## Hybrid RAG + Siamese Dual-Tower Cross-Attention + Multi-Task Research Gap Detection

**Authors:** Research & Development Team  
**System Architecture:** Production-Grade Evidence-Grounded Neural Pipeline  
**Version:** 2.0.0 (Neural Architecture)

---

### 1. Executive Summary & Research Problem
Assessing technology innovation across disjoint scientific corpora (academic literature) and industrial property (patent databases) has traditionally suffered from two major limitations:
1. **Unreliable Black-Box Generative Models:** Relying on commercial LLMs (such as GPT-4o) for relationship scoring and gap identification produces hallucinated cross-references, ungrounded technical gaps, and non-deterministic scores lacking empirical reproducibility.
2. **Disconnected Retrieval vs. Reasoning:** Classical vector search or naive ANNs either treat cross-corpus mapping as pure lexical/semantic matching or conflate document retrieval with relationship reasoning and gap discovery.

### 2. Primary Objective & Architectural Separation
This implementation establishes a mathematically grounded, verifiable, and evidence-backed framework with strict separation of responsibilities:
- **Hybrid RAG Layer (BM25 + TF-IDF + Sentence-Transformers + FAISS + RRF):** Responsible exclusively for **retrieving relevant papers and patents** without making relationship claims.
- **Siamese / Dual-Tower + Cross-Attention Model:** Responsible for **learning the deep semantic and technical relationship** between pairs of papers and patents:
  $$\mathcal{P}(\text{Related} \mid \text{Paper}, \text{Patent}) \in [0, 1]$$
- **Research Gap Detection Network:** A multi-task PyTorch neural network that evaluates cross-corpus coverage, unresolved technical limitations, combination rarity, and relationship scores to **detect, categorize, and score research gap opportunities**.
- **Innovation Scoring & Evidence Validation:** A transparent decision layer quantifying novelty, commercial viability, and verifying that every detected gap is supported by cited passages from OpenAlex and EPO OPS records.


### 2. Technology and Model Contributions

| Component | Category | Role | Key Contribution |
| :--- | :--- | :--- | :--- |
| **OpenAlex API** | Ingestion | Academic Literature Retrieval | Provides peer-reviewed research papers, DOIs, concepts, and citations |
| **EPO OPS API** | Ingestion | Industrial Patent Retrieval | Provides verified patent grants, claims, CPC/IPC classifications, and priority dates |
| **BM25** | Retrieval | Sparse Term-Frequency Matcher | Captures exact technical terminology, domain keywords, and author names |
| **TF-IDF** | Retrieval | Rare Domain Keyword Weighting | Weights discriminating sub-field terms across cross-corpus vocabularies |
| **Sentence-Transformer** | Retrieval | Dense Representation Encoder | Encodes contextual semantics into 384-dimensional dense vectors (`all-MiniLM-L6-v2`) |
| **FAISS** | Retrieval | Dense Vector Indexing | Enables sub-millisecond maximum inner-product and L2 nearest-neighbor search |
| **Reciprocal Rank Fusion (RRF)** | Retrieval | Multi-Retriever Rank Fusion | Combines sparse and dense rankings robustly: $RRF(d) = \sum \frac{1}{60 + r_m(d)}$ |
| **Dual-Tower Encoder** | Neural Architecture | Domain-Specific Representation | Separately encodes academic paper and patent claim texts with dedicated projections |
| **Cross-Attention** | Neural Architecture | Concept Interaction Learning | Bidirectional Multi-Head Attention capturing token-level interactions ($Q, K, V$) |
| **RAG Feature Fusion** | Neural Architecture | Multi-Modal Signal Fusion | Projects 10 non-leaking retrieval features into the relationship classification head |
| **Gap Detection Network** | Neural Architecture | Multi-Task Gap Identification | Neural network predicting Gap Score (BCE), Gap Category (CE), and Confidence (MSE) |
| **Evidence Validation** | Verification | Hallucination Prevention | Enforces strict provenance; discards any gap without ground-truth cited passages |
| **Innovation Scoring** | Decision Layer | Multi-Factor Opportunity Scoring | Computes composite index: $S = w_r \cdot S_{rel} + w_g \cdot S_{gap} + w_n \cdot S_{nov}$ |


### 3. Target Final Architecture

```text
                                  USER QUERY
                                       │
                         Query Analysis & Keyword Extraction
                                       │
            ┌──────────────────────────┴──────────────────────────┐
            ▼                                                     ▼
     OpenAlex API (REST)                                   EPO OPS API (OAuth2)
   [Research Literature]                                    [Patent Claims & CPC]
            │                                                     │
            └──────────────────────────┬──────────────────────────┘
                                       ▼
                          Data Preprocessing & Cleaning
                   (HTML stripping, token normalization, deduplication)
                                       │
                                       ▼
                            HYBRID RAG RETRIEVAL
            ┌──────────────────────────┼──────────────────────────┐
            ▼                          ▼                          ▼
       BM25 Engine               TF-IDF Engine          Sentence-Transformer
   (Exact keyword match)     (Rare term weighting)      (MiniLM-L6-v2 dense)
            │                          │                          │
            │                          │                          ▼
            │                          │                      FAISS Index
            └──────────────────────────┼──────────────────────────┘
                                       ▼
                       Reciprocal Rank Fusion (RRF, k=60)
                                       │
                                       ▼
                           Top-K Retrieved Papers + Patents
                                       │
                                       ▼
                           Candidate Pair Construction
                      (Top-K retrieval without Cartesian explosion)
                                       │
                 ┌─────────────────────┴─────────────────────┐
                 ▼                                           ▼
       Paper Representation                        Patent Representation
   (MiniLM + Paper Projection)                 (MiniLM + Patent Projection)
                 │                                           │
                 └─────────────────────┬─────────────────────┘
                                       ▼
                     Bidirectional Multi-Head Cross-Attention
                         (Token & concept-level interaction)
                                       │
                                       ▼
                           Learned Interaction Vector
                                       +
                           10 Scaled RAG Features
             (BM25, TF-IDF, Semantic sim, FAISS dist, RRF, Jaccard, Overlap, etc.)
                                       │
                                       ▼
                         Dual-Tower Relationship Head
                                       │
                                       ▼
                      Paper-Patent Relationship Score ∈ [0, 1]
                                       │
                                       ▼
                 ┌───────────────────────────────────────────┐
                 │    RESEARCH GAP DETECTION NETWORK (PyTorch)│
                 │                                           │
                 │  Input: 12 Corpus & Evidence Features     │
                 │  (Coverage, Rarity, Limitations, Rel-Score)│
                 │                                           │
                 │          Feature Saliency Attention       │
                 │                 Dense Layers              │
                 │                                           │
                 │  [Head 1: Gap Score] [Head 2: Gap Type]  │
                 │        [Head 3: Confidence Score]         │
                 └─────────────────────┬─────────────────────┘
                                       │
                                       ▼
                              Evidence Validation
               (Strict passage extraction, URL verification, provenance)
                                       │
                                       ▼
                         Final Innovation Assessment
                      (Exported to Dashboard & API)
```


In [ ]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Ensure backend modules are discoverable
root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root_path / "backend"))
os.chdir(str(root_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Working Directory:", Path.cwd())
print(f"PyTorch Version: {torch.__version__} | Execution Device: {device}")


### 4. API-Based Data Ingestion Pipeline
The framework interfaces directly with authoritative scholarly and patent databases:
1. **OpenAlex API:** Queries over 250M academic works, returning abstracts, DOIs, publication years, concept hierarchies, and citation links.
2. **EPO Open Patent Services (OPS) API:** Connects via OAuth2 token negotiation to fetch patent abstracts, bibliographic data, IPC/CPC classification codes, and claims.

The pipeline implements robust rate limiting, pagination, batching, and error handling without hardcoding secrets.


In [ ]:
from app.ingestion.preprocessing import TextPreprocessor

raw_papers_path = Path("data/raw/papers.json")
raw_patents_path = Path("data/raw/patents.json")

print("Checking existing collected data in data/raw/...")
if raw_papers_path.exists() and raw_patents_path.exists():
    with open(raw_papers_path, "r", encoding="utf-8") as f:
        raw_papers = json.load(f)
    with open(raw_patents_path, "r", encoding="utf-8") as f:
        raw_patents = json.load(f)
    print(f"Successfully loaded {len(raw_papers)} raw papers and {len(raw_patents)} raw patents.")
else:
    print("Collecting fresh dataset from APIs via training/collect_data.py...")
    MAX_PAPERS = 50
    MAX_PATENTS = 20
    print(f"Configured: MAX_PAPERS={MAX_PAPERS}, MAX_PATENTS={MAX_PATENTS}")


### 5. Dataset Statistics & Corpus Profiling
Below, we automatically compute and report comprehensive corpus metrics from the cleaned dataset. No statistics are hardcoded.


In [ ]:
train_df = pd.read_csv("data/training/train.csv")
val_df = pd.read_csv("data/training/validation.csv")
test_df = pd.read_csv("data/training/test.csv")
candidates_df = pd.read_csv("data/processed/candidates.csv")

total_pairs = len(train_df) + len(val_df) + len(test_df)
pos_pairs = (train_df["label"] == 1).sum() + (val_df["label"] == 1).sum() + (test_df["label"] == 1).sum()
neg_pairs = total_pairs - pos_pairs

stats_summary = {
    "Total Candidate Pairs": total_pairs,
    "Training Set Pairs": len(train_df),
    "Validation Set Pairs": len(val_df),
    "Test Set Pairs": len(test_df),
    "Total Positive Pairs": int(pos_pairs),
    "Total Negative Pairs": int(neg_pairs),
    "Positive Ratio": f"{pos_pairs / total_pairs:.2%}",
    "Unique Papers in Corpus": len(raw_papers),
    "Unique Patents in Corpus": len(raw_patents),
    "Average Paper Text Length (chars)": int(np.mean([len(p.get("abstract", "") or "") for p in raw_papers])),
    "Average Patent Text Length (chars)": int(np.mean([len(p.get("abstract", "") or "") for p in raw_patents]))
}

display_stats = pd.DataFrame(list(stats_summary.items()), columns=["Metric", "Value"])
display_stats


### 6. Hybrid RAG Retrieval Engine Demonstration
The retrieval layer combines sparse lexical matching (BM25, TF-IDF) with dense semantic search (Sentence-Transformers + FAISS) via Reciprocal Rank Fusion:
$$RRF(d) = \sum_{m \in M} \frac{1}{60 + r_m(d)}$$
This guarantees that queries containing specialized terminology (e.g., specific chemical formulas or algorithms) benefit from exact BM25/TF-IDF matches, while conceptual queries benefit from FAISS semantic embeddings.


In [ ]:
from app.retrieval.bm25 import BM25Index
from app.retrieval.tfidf import TFIDFRetriever
from app.retrieval.embeddings import SentenceTransformerEmbedding
from app.retrieval.vector_index import FAISSIndex
from app.retrieval.fusion import reciprocal_rank_fusion

# Initialize hybrid retrievers
bm25 = BM25Index()
tfidf = TFIDFRetriever()
embedder = SentenceTransformerEmbedding()
faiss_idx = FAISSIndex(dim=384)

# Format documents
corpus_docs = []
for p in raw_papers:
    corpus_docs.append({"id": p["id"], "text": f"{p['title']} {p.get('abstract', '')}", "type": "paper"})
for pt in raw_patents:
    corpus_docs.append({"id": pt["id"], "text": f"{pt['title']} {pt.get('abstract', '')}", "type": "patent"})

# Index documents
bm25.index([d["text"] for d in corpus_docs], [d["id"] for d in corpus_docs])
tfidf.fit([d["text"] for d in corpus_docs], [d["id"] for d in corpus_docs], [d["type"] for d in corpus_docs])
embeddings = embedder.embed([d["text"] for d in corpus_docs])
faiss_idx.index(embeddings, [d["id"] for d in corpus_docs])

# Test Query
test_query = "deep learning neural network attention mechanisms"
bm25_res = bm25.search(test_query, top_k=5)
tfidf_res = tfidf.search(test_query, top_k=5)
q_emb = embedder.embed([test_query])[0]
faiss_res = faiss_idx.search(q_emb, top_k=5)
fused_res = reciprocal_rank_fusion([bm25_res, tfidf_res, faiss_res], k=60)

print(f"Top-5 Hybrid RRF Retrieved Documents for Query: '{test_query}':")
for rank, (doc_id, rrf_score, dtype) in enumerate(fused_res[:5], 1):
    print(f"  Rank {rank} | Type: {dtype:<6} | RRF Score: {rrf_score:.4f} | ID: {doc_id}")


### 7. Leakage-Safe Candidate Generation & Independent Non-Circular Labeling
To avoid Cartesian $O(N \times M)$ complexity, candidate pairs are formed by retrieving top-$K$ candidate patents for each paper using the Hybrid RAG engine.

#### Non-Circular 3-Signal Ground-Truth Labeling:
To guarantee scientifically sound labels without circular leakage:
1. **Signal 1 (Direct Reference):** Paper DOI or author citations directly cross-referencing patent numbers.
2. **Signal 2 (Domain Classification Alignment):** Paper concept matches patent IPC/CPC technical classification sections.
3. **Signal 3 (Multi-Modal Consensus):** Independent lexical and semantic similarity both exceed established empirical thresholds.
A candidate pair is labeled positive ($y=1$) only if $\ge 2$ independent signals agree.

#### Leakage-Safe Paper-Grouped Splitting:
Splitting is performed by grouping on unique `paper_id`. No paper in the training split appears in validation or testing.


In [ ]:
train_papers = set(train_df["paper_id"])
val_papers = set(val_df["paper_id"])
test_papers = set(test_df["paper_id"])

print("Data Leakage Verification:")
print(f"  Overlap between Train and Val papers: {len(train_papers.intersection(val_papers))}")
print(f"  Overlap between Train and Test papers: {len(train_papers.intersection(test_papers))}")
print(f"  Overlap between Val and Test papers:   {len(val_papers.intersection(test_papers))}")
assert len(train_papers.intersection(test_papers)) == 0, "Data leakage detected!"
print("Leakage verification passed: Strict entity separation confirmed.")


### 8. Siamese / Dual-Tower + Cross-Attention Relationship Model
The relationship model learns whether an academic paper and an industrial patent possess an active technological and semantic link.

#### Architecture Components:
1. **Paper & Patent Towers:** Independent linear projection layers mapping 384-dimensional dense text embeddings into a shared latent space ($d=256$).
2. **Bi-Directional Multi-Head Cross-Attention:**
   $$A_{\text{paper} \to \text{patent}} = \operatorname{Softmax}\left(\frac{Q_{\text{paper}} K_{\text{patent}}^T}{\sqrt{d_k}}\right) V_{\text{patent}}$$
   Captures fine-grained concept interactions across the academic and patent domains.
3. **RAG Feature Fusion Projection:** Normalizes and projects 10 heterogeneous retrieval features:
   $$\mathbf{x}_{\text{rag}} = [S_{\text{bm25}}, S_{\text{tfidf}}, S_{\text{semantic}}, D_{\text{faiss}}, S_{\text{rrf}}, Jaccard, Overlap, S_{\text{title}}, \Delta_{\text{year}}, CPC_{\text{overlap}}]$$
4. **Classification Head:** Concatenates $[h_{\text{paper}}, h_{\text{patent}}, |h_{\text{paper}} - h_{\text{patent}}|, h_{\text{paper}} \odot h_{\text{patent}}, h_{\text{cross}}, \mathbf{p}_{\text{rag}}]$ through dense layers with LeakyReLU and Dropout to produce $P(\text{Related} \mid \text{Paper}, \text{Patent})$.


In [ ]:
from app.models.relationship_model import DualTowerCrossAttentionRelationshipModel
from app.features.relationship_features import RelationshipFeatureExtractor
import pickle

rel_model = DualTowerCrossAttentionRelationshipModel(
    encoder_dim=384,
    projection_dim=256,
    num_heads=4,
    rag_feature_dim=10
).to(device)

checkpoint_path = Path("models/relationship_model/relationship_model.pt")
if checkpoint_path.exists():
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    rel_model.load_state_dict(state_dict)
    rel_model.eval()
    print("Successfully loaded trained Dual-Tower Cross-Attention weights from checkpoint.")
else:
    print("Warning: Checkpoint not found, model initialized with random weights.")

with open("models/relationship_model/feature_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Sample forward pass
sample_paper_emb = torch.randn(1, 384, device=device)
sample_patent_emb = torch.randn(1, 384, device=device)
sample_rag_feats = torch.randn(1, 10, device=device)

with torch.no_grad():
    prob = rel_model(sample_paper_emb, sample_patent_emb, sample_rag_feats)
print(f"Sample Model Relationship Prediction Probability: {prob.item():.4f}")


### 9. Multi-Task Research Gap Detection Network
The Research Gap Detection Network evaluates corpus-level coverage and evidence features to detect and categorize research opportunities.

#### Multi-Task Output Heads:
- **Head 1 (Gap Score):** Continuous probability $\in [0, 1]$ indicating whether the candidate represents a genuine research gap.
- **Head 2 (Gap Type Classification):** 5-class categorical distribution:
  1. `TECHNICAL_GAP`: Unresolved performance bottlenecks or dataset dependencies.
  2. `RESEARCH_TO_APPLICATION_GAP`: High academic density with lagging patent coverage.
  3. `COMBINATION_GAP`: Underexplored technology or method intersections.
  4. `PATENT_COVERAGE_GAP`: Academic breakthroughs without industrial patent claims.
  5. `APPLICATION_GAP`: Proven methods unapplied to target industrial domains.
- **Head 3 (Confidence Score):** Regression head estimating confidence based on evidence count and citation grounding.


In [ ]:
from app.models.gap_detection_model import ResearchGapDetectionNetwork, GAP_TYPE_CLASSES

gap_model = ResearchGapDetectionNetwork(input_dim=12, num_classes=5).to(device)
gap_checkpoint = Path("models/gap_detection_model/gap_detection_model.pt")

if gap_checkpoint.exists():
    gap_state = torch.load(gap_checkpoint, map_location=device, weights_only=True)
    gap_model.load_state_dict(gap_state)
    gap_model.eval()
    print("Successfully loaded trained Research Gap Detection Network weights.")
else:
    print("Gap checkpoint not found.")

# Sample candidate forward pass
sample_candidate_feats = torch.randn(1, 12, device=device)
with torch.no_grad():
    gap_score, type_logits, confidence = gap_model(sample_candidate_feats)
    pred_type = GAP_TYPE_CLASSES[torch.argmax(type_logits, dim=-1).item()]

print("Gap Candidate Evaluation:")
print(f"  Predicted Gap Score:      {gap_score.item():.4f}")
print(f"  Predicted Gap Category:   {pred_type}")
print(f"  Model Confidence:         {confidence.item():.4f}")


### 10. Comparative Model Benchmark Evaluation
We benchmark the proposed **Dual-Tower + Cross-Attention** architecture against standard machine learning baselines trained on identical leak-free feature splits.


In [ ]:
model_comp_df = pd.read_csv("reports/model_comparison.csv")
print("=== Benchmark Evaluation: Classical Baselines vs Proposed Neural Architecture ===")
model_comp_df


### 11. Information Retrieval & Ranking Performance
To evaluate how effectively the hybrid RAG pipeline ranks candidate patent matches, we report Mean Reciprocal Rank (MRR), Precision@K, Recall@K, and NDCG@K.


In [ ]:
retrieval_df = pd.read_csv("reports/retrieval_metrics.csv")
print("=== Candidate Retrieval Ranking Performance ===")
retrieval_df


### 12. Systematic Ablation Study
We systematically ablate each neural and retrieval component to evaluate its empirical contribution to relationship prediction performance.


In [ ]:
ablation_df = pd.read_csv("reports/ablation_study.csv")
print("=== Empirical Ablation Study Results ===")
ablation_df


### 13. End-to-End Inference Pipeline Demonstration
The complete `AnalysisPipeline` coordinates all components:
1. Ingests user query.
2. Performs Hybrid RAG retrieval (BM25 + TF-IDF + Sentence-Transformers + FAISS + RRF).
3. Builds top-$K$ candidate paper-patent pairs.
4. Computes Dual-Tower + Cross-Attention relationship probabilities.
5. Identifies gap candidates and computes multi-task gap predictions.
6. Synthesizes an evidence-grounded innovation assessment with verified citations.


In [ ]:
from app.services.analysis_pipeline import AnalysisPipeline

pipeline = AnalysisPipeline(
    papers_path="data/raw/papers.json",
    patents_path="data/raw/patents.json",
    rel_model_path="models/relationship_model/relationship_model.pt",
    gap_model_path="models/gap_detection_model/gap_detection_model.pt"
)

demo_query = "Transformer self-attention optimization for low-power edge computing devices"
print(f"Executing End-to-End Analysis for: '{demo_query}'...")
result = pipeline.run_pipeline(demo_query)

print("\n" + "=" * 60)
print("INNOVATION ASSESSMENT RESULT")
print("=" * 60)
print(f"Query:                    {result['query']}")
print(f"Innovation Index:         {result['innovation_score']:.4f}")
print(f"Overall Gap Score:        {result['overall_gap_score']:.4f}")
print(f"Primary Gap Category:     {result['gap_type']}")
print(f"Evidence Grounding:       {result['evidence_validated']}")
print(f"Retrieved Papers Count:   {len(result['retrieved_papers'])}")
print(f"Retrieved Patents Count:  {len(result['retrieved_patents'])}")
print(f"Evaluated Pairs Count:    {len(result['relationship_pairs'])}")

print("\nTop Identified Cross-Corpus Relationship:")
if result['relationship_pairs']:
    top_pair = result['relationship_pairs'][0]
    print(f"  Paper:  {top_pair['paper_title']}")
    print(f"  Patent: {top_pair['patent_title']}")
    print(f"  Predicted Relationship Probability: {top_pair['relationship_score']:.4f}")

print("\nTop Supporting Evidence Passages:")
for i, ev in enumerate(result['evidence_passages'][:2], 1):
    source_info = f"{ev.get('source_type', 'corpus')} - {ev.get('id', 'N/A')}"
    passage_text = ev.get('passage', '')[:120]
    print(f"  [{i}] ({source_info}): {passage_text}...")


### 14. Conclusion & Scientific Research Positioning
This research successfully establishes an **Evidence-Grounded AI Framework for Cross-Corpus Innovation Assessment**.

#### Defensible Architectural Division:
- **RAG** $\to$ Evidence Retrieval across OpenAlex and EPO OPS corpora.
- **Dual-Tower + Cross-Attention** $\to$ Paper-Patent Relationship Learning.
- **Research Gap Detection Network** $\to$ Multi-Task Research Gap Scoring & Categorization.
- **Innovation Scoring** $\to$ Composite Opportunity Quantification.
- **Evidence Validation** $\to$ Provenance Verification and Hallucination Prevention.

All components operate deterministically with verifiable mathematical provenance, providing a reproducible alternative to opaque generative LLM pipelines.
